In [ ]:
#| default_exp regularize.group_regularize_callback

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
from copy import copy

from fastai.callback.core import Callback
from fastai.callback.fp16 import MixedPrecision, NonNativeMixedPrecision
from torch.amp import GradScaler
from fasterai.core.schedule import Schedule
from fasterai.prune.pruner import Pruner
from fasterai.prune.prune_callback import PruneCallback

## Overview

`GroupRegularizeCallback` calls `Pruner.regularize()` at every optimizer step of a fastai fit: the gradients receive the [group penalty](../prune/pruner.html#group-sparse-learning) of a `Pruner` built with `reg > 0`, so training pushes the coupled channels the pruner will remove (a convolution's filters, its BatchNorm scale, the input slices of the layers that read it) towards zero. The same `Pruner` then removes them with `prune_model()`, once the fit is over.

In [ ]:
#| export
class GroupRegularizeCallback(Callback):
    "Add `pruner`'s group penalty to the gradients at every optimizer step, so training pushes coupled channels towards zero before `pruner.prune_model()` removes them"
    order = MixedPrecision.order - 2  # after GradientAccumulation skips a batch, before MixedPrecision unscales
    def __init__(self,
                 pruner: Pruner,                    # Built with `reg > 0` on the model the Learner trains
                 schedule: Schedule | None = None,  # Ramps the penalty: `pruner.reg * progress`
                 verbose: bool = False,             # Print the penalty after each epoch
    ):
        if not pruner.reg: raise ValueError("GroupRegularizeCallback needs a penalty: build the Pruner with reg > 0.")
        self.pruner, self.schedule, self.verbose, self.scale = pruner, copy(schedule), verbose, 1.

    def before_fit(self):
        if self.pruner.model is not self.learn.model: raise ValueError("The Pruner was built on another model than learn.model: build it on learn.model.")
        if any(isinstance(cb, NonNativeMixedPrecision) for cb in self.learn.cbs): raise ValueError("NonNativeMixedPrecision is not supported: use MixedPrecision (learn.to_fp16()).")
        if any(isinstance(cb, PruneCallback) for cb in self.learn.cbs): raise ValueError("PruneCallback cannot run in the same fit: fit with GroupRegularizeCallback, then call pruner.prune_model().")

    def before_step(self):
        self.scale = self.schedule.progress(self.pct_train) if self.schedule is not None else 1.
        if not self.scale: return
        scaler = getattr(self.learn, 'scaler', None)
        loss_scale = scaler.get_scale() if isinstance(scaler, GradScaler) else 1.
        self.pruner.regularize(scale=self.scale * loss_scale)

    def after_epoch(self):
        if self.verbose: print(f"Group penalty: {self.pruner.reg * self.scale:.2e}")

In [ ]:
show_doc(GroupRegularizeCallback)

---

[source](https://github.com/FasterAI-Labs/fasterai/blob/master/fasterai/regularize/group_regularize_callback.py#L18){target="_blank" style="float:right; font-size:smaller"}

### GroupRegularizeCallback

```python
def GroupRegularizeCallback(
    pruner:Pruner, # Built with `reg > 0` on the model the Learner trains
    schedule:Schedule | None=None, # Ramps the penalty: `pruner.reg * progress`
    verbose:bool=False, # Print the penalty after each epoch
):
```

*Add `pruner`'s group penalty to the gradients at every optimizer step, so training pushes coupled channels towards zero before `pruner.prune_model()` removes them*

## Usage Example

Build the `Pruner` on `learn.model` with `reg > 0`, train with the callback, prune, then fine-tune:

```python
from fasterai.core.criteria import large_final
from fasterai.core.schedule import lin
from fasterai.prune.pruner import Pruner
from fasterai.regularize.group_regularize_callback import GroupRegularizeCallback

xb, _ = learn.dls.one_batch()
pruner = Pruner(learn.model, 0.3, 'local', large_final, reg=1e-4, example_inputs=xb)
learn.fit(10, cbs=GroupRegularizeCallback(pruner, schedule=lin))
pruner.prune_model()
learn.fit(2, reset_opt=True)
```

- The prune replaces the parameters of every layer it shrinks, so the next fit needs `reset_opt=True`: without it, fastai keeps the optimizer it built on the old parameters.
- It works under `learn.to_fp16()` and `GradientAccumulation`: the penalty is multiplied by the loss scale before `MixedPrecision` unscales the gradients, and added once per optimizer step, not once per batch. `NonNativeMixedPrecision` is refused.
- Frozen layers get no penalty.
- Prune after the fit, not during: `PruneCallback`, which prunes during training, is refused in the same fit.

---

## See Also

- [Pruner](../prune/pruner.html) - `reg`, `alpha` and `regularize()`, the penalty this callback applies
- [PruneCallback](../prune/prune_callback.html) - Prune during training instead of after it
- [RegularizeCallback](regularize_callback.html) - Penalize each layer's own weights, one layer at a time
- [Schedules](../core/schedules.html) - Ramp the penalty over training

Tests live in `nbs/tests/test_group_regularize_callback.ipynb`.